# Protocol 5 — Causal TMS: Insula/dlPFC

This notebook simulates the key predictions of Protocol 5
(*Causal Neuromodulation Study: Insular and Frontoparietal Gating of Ignition Threshold via Dissociable Mechanisms*)
— Pred 5.A–Pred 5.D:

* **Pred 5.A** — Anterior insula (aINS) and frontoparietal TMS (dlPFC/PPC) both reduce PCI via dissociable mechanisms: HEP–PCI coupling abolished under aINS TMS but preserved under dlPFC TMS (BF₀₁ ≥ 6)
* **Pred 5.B** — aINS TMS selectively abolishes HEP–P3b coupling (coupling coefficient < 0.15); exteroceptive P3b spared; dlPFC TMS leaves HEP unaffected
* **Pred 5.C** — High baseline IA participants show strongest aINS TMS effects on PCI (accuracy × TMS-site interaction p < 0.05, η² > 0.10); absent for dlPFC site
* **Pred 5.D** — High-Πⁱ individuals show larger PCI decreases following aINS TMS than low-Πⁱ; absent for dlPFC TMS site

> All data are synthetic. Parameters come from `protocols/protocol_5_causal_tms.json`.
>
> **EP-5 | Paper 2 | Empirical | Depends on: EP-0 | N target = 36, N minimum = 28**
>
> Primary aINS stimulation modality: tFUS (transcranial focused ultrasound). dlPFC/PPC and vertex use standard figure-of-eight TMS coil.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from apgi.core import compute_pi_i_eff, compute_S_t, ignition_criterion, theta_equilibrium, BETA_SM_DEFAULT

## 1 — Protocol 5 parameters

In [ ]:
# From protocol_5_causal_tms.json → apgi_parameters
# EP-5: Paper 2 — aINS (tFUS) and dlPFC/PPC (TMS) as the two active stimulation sites
KAPPA = 100.0
ALPHA = 0.3
BETA = 0.7
# Delta Πⁱ per stimulation site (from protocol_5_causal_tms.json apgi_parameters)
DELTA_PI = {
    "vertex_sham": 0.0,       # active sham — no Πⁱ effect
    "ains_active": -0.4,       # aINS tFUS: selectively reduces Πⁱ_eff (interoceptive gating)
    "dlpfc_ppc_active": -0.2,  # dlPFC/PPC TMS: global threshold shift, HEP unaffected
}
N_TRIALS = 480
PI_I_BASE = 1.0

## 2 — Simulate three TMS conditions

In [ ]:
def run_condition(delta_pi, seed):
    rng = np.random.default_rng(seed)
    pi_i = max(0.01, PI_I_BASE + delta_pi)
    C_metabolic = rng.uniform(0.5, 2.0, N_TRIALS)
    M_hat = rng.uniform(0.0, 0.5, N_TRIALS)
    pi_i_eff = np.array([compute_pi_i_eff(pi_i, BETA_SM_DEFAULT, float(M_hat[t])) for t in range(N_TRIALS)])
    pi_e = rng.uniform(0.8, 1.5, N_TRIALS)
    z_e = rng.uniform(0.2, 1.0, N_TRIALS)
    z_i = rng.uniform(0.1, 0.8, N_TRIALS)
    V_info = rng.uniform(0.1, 1.0, N_TRIALS)
    S_t = np.array(
        [
            compute_S_t(
                float(pi_e[t]), float(z_e[t]), float(pi_i_eff[t]), float(z_i[t])
            )
            for t in range(N_TRIALS)
        ]
    )
    theta_t = np.array(
        [
            theta_equilibrium(float(C_metabolic[t]), float(V_info[t]), kappa_meta=ALPHA, delta_info=BETA)
            for t in range(N_TRIALS)
        ]
    )
    detected = np.array(
        [ignition_criterion(S_t[t], theta_t[t]) for t in range(N_TRIALS)]
    )
    hep = pi_i_eff + rng.normal(0, 0.08, N_TRIALS)
    p3b = S_t * detected + rng.normal(0, 0.05, N_TRIALS)
    pci = float(detected.mean() * 0.8 + rng.uniform(0, 0.05))
    return dict(detected=detected, hep=hep, p3b=p3b, pci=pci)


conditions = {
    lbl: run_condition(dp, seed=i + 1) for i, (lbl, dp) in enumerate(DELTA_PI.items())
}
for lbl, c in conditions.items():
    r, _ = pearsonr(c["hep"], c["p3b"])
    print(
        f"{lbl:<20} ignition={c['detected'].mean():.2f}  PCI={c['pci']:.3f}  HEP-P3b r={r:.3f}"
    )

## 3 — Visualise predictions Pred 5.A–Pred 5.B

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels = list(conditions.keys())
colors = ["#2166ac", "#d6604d", "#9966FF"]

# Pred 5.A: PCI by condition
ax = axes[0]
ax.bar(
    labels,
    [conditions[k]["pci"] for k in labels],
    color=colors,
    alpha=0.85,
    edgecolor="white",
    width=0.4,
)
ax.axhline(0.31, ls="--", lw=1, color="k", alpha=0.6, label="PCI threshold")
ax.set_title("Pred 5.A — PCI by TMS site\n(aINS + dlPFC/PPC reduce PCI vs vertex)")
ax.set_ylabel("PCI")
ax.set_ylim(0, 0.8)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

# Pred 5.B: Ignition rate
ax = axes[1]
ax.bar(
    labels,
    [conditions[k]["detected"].mean() for k in labels],
    color=colors,
    alpha=0.85,
    edgecolor="white",
    width=0.4,
)
ax.set_title("Pred 5.B — Ignition rate\n(aINS TMS selectively disrupts interoceptive gating)")
ax.set_ylabel("Ignition rate")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)

# Pred 5.B: HEP-P3b coupling
ax = axes[2]
r_vals = [pearsonr(conditions[k]["hep"], conditions[k]["p3b"])[0] for k in labels]
ax.bar(labels, r_vals, color=colors, alpha=0.85, edgecolor="white", width=0.4)
ax.axhline(0.15, ls="--", lw=1, color="k", alpha=0.6, label="coupling threshold 0.15")
ax.axhline(0, ls="-", lw=0.6, color="k", alpha=0.3)
ax.set_title("Pred 5.B — HEP–P3b coupling\n(aINS TMS: coupling < 0.15)")
ax.set_ylabel("Pearson r")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 5 — Causal TMS: Insula/dlPFC (Pred 5.A–Pred 5.D)", fontsize=12)
plt.tight_layout()
plt.show()

## 4 — APGI parameter interpretation

The key dissociation (Pred 5.A–5.B): **aINS tFUS** lowers $\Pi^i_{\text{eff}}$ by Δ = −0.4,
which selectively abolishes HEP–P3b coupling (interoceptive gating stream) while sparing
exteroceptive P3b. **dlPFC/PPC TMS** (Δ = −0.2) reduces global ignition probability without
affecting HEP. Vertex sham provides the active control baseline.

This maps onto APGI as:
$$\Pi^i_{\text{eff, TMS}} = \Pi^i \cdot e^{-C/\kappa} \cdot e^{\Delta\Pi^i_{\text{TMS}}}$$

| TMS Site | $\Delta\Pi^i$ | Predicted effect |
|---|---|---|
| Vertex (sham) | 0.0 | No change |
| aINS (tFUS) | −0.4 | Selective interoceptive gating disruption; HEP–PCI coupling abolished |
| dlPFC/PPC (TMS) | −0.2 | Global threshold shift; HEP unaffected (BF₀₁ ≥ 6) |

**Pred 5.C/5.D:** High-IA / high-Πⁱ participants show larger PCI reductions under aINS tFUS
than low-IA participants (accuracy × TMS-site interaction p < 0.05, η² > 0.10); this
interaction is absent for the dlPFC/PPC site.